# Quickstart: Wavelength Selection with spectral_select

This notebook demonstrates the basic workflow for wavelength selection analysis using the `spectral_select` library.

**What you'll learn:**
- Load preprocessed hyperspectral data
- Configure and run wavelength selection analysis
- Extract selected wavelength bands
- Create basic visualizations

## 1. Setup

Import the core classes from `spectral_select`:

In [1]:
from spectral_select import Analyzer, Config, SpectraData, Visualizer

## 2. Load Data

Load preprocessed hyperspectral data from a pickle file. The data should have been preprocessed using the standard pipeline.

In [2]:
from pathlib import Path

# Get project root (assuming notebook is in notebooks/examples/)
project_root = Path.cwd().parent.parent

# Data path - update this for your data file
DATA_PATH = project_root / "Data" / "processed" / "LichensProcessed" / "data_cutoff_60nm_exposure_max_power_min.pkl"

# Verify data file exists
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Data file not found: {DATA_PATH}\nRun 00_data_loading.ipynb first to process raw data.")

data = SpectraData.from_pickle(DATA_PATH)

# SpectraData is a multi-excitation container
print(f"Sample: {data.sample_name}")
print(f"Spatial shape: {data.spatial_shape}")
print(f"Number of excitations: {data.n_excitations}")
print(f"Excitation wavelengths: {data.excitation_wavelengths}")

# Access per-excitation data
print("\nPer-excitation details:")
for ex_nm in data.excitation_wavelengths:
    ex_data = data.get_excitation(ex_nm)
    print(f"  Ex {ex_nm:.0f}nm: {ex_data.n_bands} bands, shape {ex_data.shape}")

Sample: data_cutoff_60nm_exposure_max_power_min
Spatial shape: (256, 348)
Number of excitations: 8
Excitation wavelengths: [310.0, 325.0, 340.0, 365.0, 385.0, 400.0, 415.0, 430.0]

Per-excitation details:
  Ex 310nm: 18 bands, shape (256, 348, 18)
  Ex 325nm: 18 bands, shape (256, 348, 18)
  Ex 340nm: 20 bands, shape (256, 348, 20)
  Ex 365nm: 24 bands, shape (256, 348, 24)
  Ex 385nm: 26 bands, shape (256, 348, 26)
  Ex 400nm: 27 bands, shape (256, 348, 27)
  Ex 415nm: 27 bands, shape (256, 348, 27)
  Ex 430nm: 27 bands, shape (256, 348, 27)


## 3. Configure Analysis

Create a configuration with your analysis parameters. The `Config` class provides sensible defaults for most parameters.

In [3]:
config = Config(
    sample_name="LichensProcessed",
    n_bands_to_select=10,
    device="cpu",  # Use "cuda" if GPU available
    training_epochs=20,  # Use fewer epochs for faster execution
)

print(f"Sample: {config.sample_name}")
print(f"Bands to select: {config.n_bands_to_select}")
print(f"Device: {config.device}")
print(f"Training epochs: {config.training_epochs}")

Sample: LichensProcessed
Bands to select: 10
Device: cpu
Training epochs: 20


## 4. Run Analysis

Create an `Analyzer` and fit it to the data. This runs the full wavelength selection pipeline:
1. Train autoencoder on spectral data
2. Extract baseline representations
3. Perturb input dimensions and measure influence
4. Rank wavelengths by influence score

In [4]:
analyzer = Analyzer(config)
analyzer.fit(data)

print("Analysis complete!")
print(f"Selected bands: {analyzer.result.n_bands}")
print(f"Compression ratio: {analyzer.result.metrics.compression_ratio:.1f}x")

Preparing data for 8 excitation wavelengths...
Emission band lengths for each excitation wavelength:
  - Excitation 310.0 nm: 18 bands
  - Excitation 325.0 nm: 18 bands
  - Excitation 340.0 nm: 20 bands
  - Excitation 365.0 nm: 24 bands
  - Excitation 385.0 nm: 26 bands
  - Excitation 400.0 nm: 27 bands
  - Excitation 415.0 nm: 27 bands
  - Excitation 430.0 nm: 27 bands
No mask provided. All pixels considered valid.


Training new autoencoder model. This may take several minutes...


Global data range (valid values only): [0.0000, 120941.2608]
Data normalized to range [0, 1] using global normalization
Data preparation complete. Spatial dimensions: 256x348


/Users/narekmeloyan/PycharmProjects/4D-Hyperspectral-Unsupervised-Clustering/.venv/lib/python3.11/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Creating spatial chunks for each excitation wavelength...
Created 35 chunks for each excitation
Starting training for 20 epochs with 35 batches...


...

Epoch 17/20, Loss: 0.0010 (Recon: 0.0005, Sparsity: 0.0005), Time: 6.27s
  New best model saved to model_output/best_hyperspectral_model.pth (loss: 0.0010)


Epoch 18/20, Loss: 0.0010 (Recon: 0.0005, Sparsity: 0.0005), Time: 6.28s
  No improvement for 1 epochs (best: 0.0010 at epoch 17)


Epoch 19/20, Loss: 0.0010 (Recon: 0.0005, Sparsity: 0.0005), Time: 6.41s
  No improvement for 2 epochs (best: 0.0010 at epoch 17)


Epoch 20/20, Loss: 0.0010 (Recon: 0.0005, Sparsity: 0.0005), Time: 6.29s
  New best model saved to model_output/best_hyperspectral_model.pth (loss: 0.0010)
Final model saved to model_output/final_hyperspectral_model.pth
Training completed. Best loss: 0.0010 at epoch 20


Analysis complete!
Selected bands: 10
Compression ratio: 18.7x


## 5. Get Results

Extract the selected wavelength bands. Each band includes excitation/emission wavelengths, influence score, and rank.

In [5]:
bands = analyzer.get_wavelengths()

print(f"Selected {len(bands)} wavelength bands:\n")

# Show top 10 bands
for band in bands[:10]:
    print(f"Rank {band.rank:2d}: Ex={band.excitation_nm:3.0f}nm, Em={band.emission_nm:3.0f}nm, Score={band.influence_score:.4f}")

Selected 10 wavelength bands:

Rank  1: Ex=310nm, Em=420nm, Score=0.3800
Rank  2: Ex=310nm, Em=430nm, Score=0.1375
Rank  3: Ex=340nm, Em=420nm, Score=0.0639
Rank  4: Ex=310nm, Em=440nm, Score=0.0443
Rank  5: Ex=310nm, Em=720nm, Score=0.0425
Rank  6: Ex=340nm, Em=430nm, Score=0.0385
Rank  7: Ex=310nm, Em=690nm, Score=0.0334
Rank  8: Ex=365nm, Em=430nm, Score=0.0330
Rank  9: Ex=310nm, Em=710nm, Score=0.0327
Rank 10: Ex=310nm, Em=700nm, Score=0.0265


## 6. Visualize Results

Create visualizations using the `Visualizer` class. The factory method `from_analyzer` automatically extracts the needed data.

In [6]:
viz = Visualizer.from_analyzer(analyzer)

In [7]:
# Influence heatmap shows score distribution across excitation/emission grid
viz.plot_influence_heatmap()

PosixPath('visualizations/LichensProcessed/influence_heatmap.png')

In [8]:
# Scatter plot shows selected wavelengths with size indicating rank
viz.plot_wavelength_scatter()

PosixPath('visualizations/LichensProcessed/wavelength_scatter.png')

In [9]:
# Ranking bar chart shows influence scores by rank
viz.plot_influence_ranking()

PosixPath('visualizations/LichensProcessed/influence_ranking.png')

## Summary

In this notebook, you learned how to:

1. **Load data** with `SpectraData.from_pickle()`
2. **Configure** analysis with `Config()`
3. **Run analysis** with `Analyzer.fit()`
4. **Extract results** with `Analyzer.get_wavelengths()`
5. **Visualize** with `Visualizer.from_analyzer()`

**Next steps:**
- See `02_validation.ipynb` for ground truth validation
- Explore different `Config` parameters
- Try different datasets